In [1]:
import numpy as np
from aeon.classification import BaseClassifier
from aeon.transformations.collection.convolution_based import MultiRocket
from aeon.transformations.collection.convolution_based._hydra import HydraTransformer
from aeon.utils.validation import check_n_jobs
from aeon.transformations.collection.interval_based import QUANTTransformer
import numpy as np
import polars as pl
from aeon.classification.base import BaseClassifier
from aeon.classification.feature_based import (
    Catch22Classifier,
)
import os
from aeon.transformations.collection.convolution_based import Rocket
from aeon.datasets.tsc_datasets import univariate
from sklearn.base import clone
from aeon.classification.convolution_based import MultiRocketHydraClassifier
from aeon.classification.convolution_based import RocketClassifier
from sklearn.metrics import accuracy_score
from aeon.classification.interval_based import QUANTClassifier
from autotsc import utils, models
from tqdm import tqdm
from aeon.classification.feature_based import Catch22Classifier
from aeon.classification.interval_based import QUANTClassifier
from aeon.classification.shapelet_based import RDSTClassifier


2025-12-04 14:25:56.391758: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
class CrossValidationWrapper(BaseClassifier):
    def __init__(self, model):
        super().__init__()
        self.model = model
        self.trained_models_ = []
        self.fit_time_ = []
        self.fit_time_mean_ = None
        self.predict_time_ = []
        self.predict_time_mean_ = None

    def _fit(self, X, y):
        raise NotImplementedError()

    def _predict_proba(self, X):
        predictions = []
        for model in self.trained_models_:
            proba = model.predict_proba(X)
            predictions.append(proba)
        avg_proba = np.mean(predictions, axis=0)
        return avg_proba

    def _predict(self, X):
        probas = self._predict_proba(X)
        predicted_indices = np.argmax(probas, axis=1)
        return self.classes_[predicted_indices]

    def get_all_oof_proba(self):
        return pl.DataFrame(self.oof_proba).sort('index', maintain_order=True)

    def _fit_predict_proba(self, X, y, cv_splits):
        self.oof_proba = []
        for train_idx, val_idx in tqdm(cv_splits):
            model_clone = clone(self.model)
            X_train, y_train = X[train_idx], y[train_idx]
            X_valid, _ = X[val_idx], y[val_idx]
            model_clone.fit(X_train, y_train)
            self.trained_models_.append(model_clone)
            proba = model_clone.predict_proba(X_valid)
            prob_columns = []
            for idx, p in zip(val_idx, proba):
                d = {
                    'index': idx,
                }
                for scls, prob in zip(model_clone.classes_, p):
                    k = f'proba_class_{scls}'
                    d[k] = prob.item()
                    if k not in prob_columns:
                        prob_columns.append(k)
                self.oof_proba.append(d)
        return pl.DataFrame(self.oof_proba).group_by('index').mean().sort('index').select(prob_columns).to_numpy()

    def fit_predict_proba(self, X, y, cv_splits):
        X, y, single_class = self._fit_setup(X, y)
        y_proba = self._fit_predict_proba(X, y, cv_splits)
        self.is_fitted = True
        return y_proba

In [3]:
def generate_folds(X, y, n_splits=5, n_repetitions=5, random_state=0):
    all_folds = []
    for i in range(n_repetitions):
        folds = utils.get_folds(X, y, n_splits=n_splits, random_state=random_state+i)
        all_folds.extend(folds)
    return all_folds

import numpy as np
from scipy.interpolate import interp1d
from aeon.transformations.collection.base import BaseCollectionTransformer


class DownsampleTransformer(BaseCollectionTransformer):
    _tags = {
        "X_inner_type": ["np-list", "numpy3D"],
        "capability:multivariate": True,
        "capability:unequal_length": True,
        "fit_is_empty": True,
    }

    def __init__(self, proportion):
        self.proportion = proportion
        super().__init__()

    def _transform(self, X, y=None):
        self._check_parameters()

        is_np = isinstance(X, np.ndarray)
        out = []

        for x in X:
            c, t = x.shape
            new_t = max(2, int(round(t * self.proportion)))

            old_grid = np.linspace(0, 1, t)
            new_grid = np.linspace(0, 1, new_t)

            xr = np.zeros((c, new_t))
            for i in range(c):
                f = interp1d(old_grid, x[i], kind="linear")
                xr[i] = f(new_grid)

            out.append(xr)

        return np.asarray(out) if is_np else out

    def _check_parameters(self):
        if not (0 < self.proportion < 1):
            raise ValueError("proportion must be between 0 and 1")


In [4]:
dataset = 'FordB'
X_train, y_train, X_test, y_test = utils.load_dataset(dataset)
from aeon.benchmarking import resampling
X_train, y_train, X_test, y_test = resampling.stratified_resample_data(X_train, y_train, X_test, y_test, random_state=0)
t= DownsampleTransformer(proportion=0.25)
X_train = t.fit_transform(X_train)
X_test = t.transform(X_test)
print(X_train.shape, X_test.shape)

all_folds = generate_folds(X_train, y_train, n_splits=10, n_repetitions=5, random_state=0)

(3636, 1, 125) (810, 1, 125)


In [5]:
m1 = CrossValidationWrapper(MultiRocketHydraClassifier(n_jobs=-1, random_state=0))
prob_hr = m1.fit_predict_proba(X_train, y_train, all_folds)

100%|██████████| 50/50 [32:23<00:00, 38.87s/it]


In [6]:
m2 = CrossValidationWrapper(QUANTClassifier(random_state=0))
prob_quant = m2.fit_predict_proba(X_train, y_train, all_folds)

100%|██████████| 50/50 [07:10<00:00,  8.60s/it]


In [7]:
m3 = CrossValidationWrapper(RDSTClassifier(random_state=0, n_jobs=-1))
prob_rdst = m3.fit_predict_proba(X_train, y_train, all_folds)

100%|██████████| 50/50 [27:05<00:00, 32.51s/it]


In [8]:
def add_argmax_label(df: pl.DataFrame, label_col="label"):
    numeric_cols = [c for c in df.columns if df[c].dtype.is_numeric()]

    return df.with_columns(
        pl.struct(numeric_cols)
        .map_elements(lambda row: max(row, key=row.get))
        .alias(label_col)
    )

pm1 = pl.DataFrame(prob_hr, schema=list(m1.classes_)).pipe(add_argmax_label)["label"].to_list()
pm1_acc = accuracy_score(y_train, pm1)
print(f"Hydra-multirocket OOF accuracy: {pm1_acc:.4f}")

pm2 = pl.DataFrame(prob_quant, schema=list(m2.classes_)).pipe(add_argmax_label)["label"].to_list()
pm2_acc = accuracy_score(y_train, pm2)
print(f"QUANT OOF accuracy: {pm2_acc:.4f}")

pm3 = pl.DataFrame(prob_rdst, schema=list(m3.classes_)).pipe(add_argmax_label)["label"].to_list()
pm3_acc = accuracy_score(y_train, pm3)
print(f"RDST OOF accuracy: {pm3_acc:.4f}")

Hydra-multirocket OOF accuracy: 0.9348
QUANT OOF accuracy: 0.9147
RDST OOF accuracy: 0.9249


In [9]:
pred_hr = m1.predict(X_test)
acc_hr = accuracy_score(y_test, pred_hr)
acc_hr

0.9296296296296296

In [10]:
pred_quant = m2.predict(X_test)
acc_quant = accuracy_score(y_test, pred_quant)
acc_quant

0.9271604938271605

In [11]:
pred_rdst = m3.predict(X_test)
acc_rdst = accuracy_score(y_test, pred_rdst)
acc_rdst

0.9333333333333333

In [17]:
m = MultiRocketHydraClassifier(n_jobs=-1, random_state=0)
m.fit(X_train, y_train)
pred = m.predict(X_test)
acc = accuracy_score(y_test, pred)
acc

0.9320987654320988

In [18]:
m = QUANTClassifier(random_state=0)
m.fit(X_train, y_train)
pred = m.predict(X_test)
acc = accuracy_score(y_test, pred)
acc

0.9234567901234568

In [19]:
m = RDSTClassifier(n_jobs=-1, random_state=0)
m.fit(X_train, y_train)
pred = m.predict(X_test)
acc = accuracy_score(y_test, pred)
acc

0.937037037037037

In [12]:
m1_test = m1.predict_proba(X_test)
m2_test = m2.predict_proba(X_test)
m3_test = m3.predict_proba(X_test)

In [13]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import RidgeClassifierCV

mm = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", RidgeClassifierCV(alphas=np.logspace(-4, 4, 10)))
])

mm.fit(np.hstack([prob_hr, prob_quant, prob_rdst]), y_train)
pred_stack = mm.predict(np.hstack([m1_test, m2_test, m3_test]))
acc_stack = accuracy_score(y_test, pred_stack)
acc_stack

0.9395061728395062

In [14]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import RidgeClassifierCV
from sklearn.ensemble import RandomForestClassifier
mm = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", RandomForestClassifier(n_jobs=-1, n_estimators=500))
])

mm.fit(np.hstack([prob_hr, prob_quant, prob_rdst]), y_train)
pred_stack = mm.predict(np.hstack([m1_test, m2_test, m3_test]))
acc_stack = accuracy_score(y_test, pred_stack)
acc_stack

0.9222222222222223

In [15]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import RidgeClassifierCV
from sklearn.ensemble import RandomForestClassifier
from catboost import CatBoostClassifier

mm = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", CatBoostClassifier(
        verbose=False,
        learning_rate=0.005
    ))
])

mm.fit(np.hstack([prob_hr, prob_quant, prob_rdst]), y_train)
pred_stack = mm.predict(np.hstack([m1_test, m2_test, m3_test]))
acc_stack = accuracy_score(y_test, pred_stack)
acc_stack

0.9358024691358025

In [16]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC

mm = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LinearSVC())
])

mm.fit(np.hstack([prob_hr, prob_quant, prob_rdst]), y_train)
pred_stack = mm.predict(np.hstack([m1_test, m2_test, m3_test]))
acc_stack = accuracy_score(y_test, pred_stack)
acc_stack

0.9395061728395062